In [ ]:
import os, sys, glob, json, zipfile, shutil, importlib
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/sketchy_dataset_one_sketch"; GOV_DIR = f"{DRIVE_ROOT}/governance"; GOV = f"{GOV_DIR}/governance.pt"
REPO_DIR, REPO_URL, BRANCH = "/content/repo", "https://github.com/Harshitha-282007/IE643_final.git", "governed-pipeline"
if not os.path.isdir(REPO_DIR):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} checkout -- .
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, f"{REPO_DIR}/src")
LOCAL = "/content/pilot_v1"
if not glob.glob(f"{LOCAL}/**/metadata.jsonl", recursive=True):
    os.makedirs(LOCAL, exist_ok=True); zipfile.ZipFile(f"{DRIVE_ROOT}/pilot_v1.zip").extractall(LOCAL)
DATA_DIR = os.path.dirname(glob.glob(f"{LOCAL}/**/metadata.jsonl", recursive=True)[0])
BASE_ROWS = [json.loads(l) for l in open(f"{DATA_DIR}/metadata.jsonl") if l.strip()]
!pip uninstall -y torchao -q
!pip install -q -U diffusers transformers accelerate safetensors peft lpips opencv-python-headless
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from PIL import Image
from pipeline import Pipeline, seed_for
import train_role_lora as TL
P = Pipeline(res=512, steps=20)
print("ready | rows:", len(BASE_ROWS))

In [ ]:
from rolelora.lora import RoleLoRA
lora = RoleLoRA.load(f"{GOV_DIR}/role_lora_run1/role_lora.pt", P.unet)
print("loaded", len(lora.names), "projections")

In [ ]:
import eval_roles as ER
from governance.evalkit import EvalKit
from rolegate.gates import RoleGates
rows = ER.eval_rows(BASE_ROWS, split="test", n_photos=12)
print(len(rows), "rows:", pd.Series([ER.kind_of(r) for r in rows]).value_counts().to_dict())
op = lambda r, k: Image.open(f"{DATA_DIR}/{r[k]}")
P.prepare_object_dirs([r.get("ref_object") or "object" for r in rows])
items = {}
for i in range(0, len(rows), 8):
    part = rows[i:i + 8]
    for r, it in zip(part, P.make_items([dict(key=r["id"], sketch=op(r, "sketch"), ref=op(r, "reference"), caption=r["caption"], obj=r["category"],
                                              seed=seed_for(r["original_id"]), ref_object=r.get("ref_object")) for r in part])):
        it.pop("S_gh", None); items[r["id"]] = it
gates = RoleGates.for_unet(P.unet, P.controlnet_res_channels())
IP_RULE = 1.0                                     # reference strength of the rule-based arm (boost test: x2 moved colour toward the reference)
def a_fixed(it, r): lora.set_scale(0); return P.run(it, shaped=False)                                    # the baseline: constant strengths
def a_rule(it, r): lora.set_scale(0); return P.run_role(it, gates, [0, 1, 1], knock=("text_colour", "text_material"), cn_text="mix", p=dict(ip=IP_RULE))[0]   # hand-fed rule
def a_lora(it, r): lora.set_scale(1); return P.run_role(it, gates, [0, 0, 0])[0]                         # learned: nothing neutralised, no detector
def a_lora_c(it, r): lora.set_scale(1, cond_only=True); return P.run_role(it, gates, [0, 0, 0])[0]                  # conditional half only
ARMS = {"fixed": a_fixed, "rule": a_rule, "lora": a_lora, "lora_cond": a_lora_c}
EVAL_DIR = "/content/role_eval"; os.makedirs(EVAL_DIR, exist_ok=True)

In [ ]:
# remove the hatched / saturated artefacts? Same rows, LoRA off vs both halves vs cond-only, one picture. Needs `lora` from cell 2 (or load role_lora.pt).
def a_lora_c(it, r): lora.set_scale(1, cond_only=True); return P.run_role(it, gates, [0, 0, 0])[0]
probe = [r for r in rows if ER.kind_of(r) in ("col", "mat")][:3]
fig, ax = plt.subplots(len(probe), 4, figsize=(11, 2.8 * len(probe)), squeeze=False)
for i, r in enumerate(probe):
    it = items[r["id"]]
    for j, (nm, fn) in enumerate([("reference", None), ("LoRA off", a_fixed), ("LoRA both halves", a_lora), ("LoRA cond-only", a_lora_c)]):
        ax[i, j].imshow(op(r, "reference") if fn is None else fn(it, r)); ax[i, j].set_title(nm, fontsize=7); ax[i, j].axis("off")
plt.tight_layout(); plt.savefig(f"{EVAL_DIR}/cond_only_check.png", dpi=110); plt.show()
lora.set_scale(1)

In [ ]:
ER.generate(rows, items, ARMS, EVAL_DIR)

In [ ]:
kit = EvalKit(rule="reference")
scores = ER.score(kit, rows, ARMS, EVAL_DIR, DATA_DIR)
comp, shape = ER.tables(scores, list(ARMS))
print("ROLE COMPLIANCE (higher is better; in brackets lower is better)"); print(comp.to_string(index=False))
print("\nSKETCH / FIDELITY"); print(shape.to_string(index=False))
comp.to_csv(f"{EVAL_DIR}/compliance.csv", index=False); shape.to_csv(f"{EVAL_DIR}/shape.csv", index=False)